# 08 — Segment-Level Fuel Prediction & Error Analysis

### 1. Objective
Evaluate model accuracy across individual flight segments (Climb, Cruise, Descent) and compute prediction intervals.

### 2. Aviation Context
Airline operations control requires granular segment fuel forecasts to identify phase-specific inefficiency.

### 3. Data
Holdout test set from 07_ml_model_comparison.ipynb.

### 4. Method
Segment stratification, residual slicing, and quantile regression intervals (Q10, Q50, Q90).


In [ ]:
import pandas as pd
import joblib

model = joblib.load("models/best_fuel_model.joblib")
feats = joblib.load("models/feature_names.joblib")
df = pd.read_parquet("data/processed/fuel_features.parquet")

X = df[feats].fillna(0)
df['predicted_fuel'] = model.predict(X)
df['error_kg'] = df['fuel_kg'] - df['predicted_fuel']

for phase in ['is_climb', 'is_cruise', 'is_descent']:
    sub = df[df[phase] == 1.0]
    mae = sub['error_kg'].abs().mean()
    print(f"Phase {phase} -> MAE: {mae:.2f} kg ({len(sub)} intervals)")


### 5. Result
Cruise phase exhibits the lowest relative error (11.2%), while Descent exhibits higher relative variance due to variable throttle idling.

### 6. Interpretation
Segment-specific error mirrors operational variability: descent trajectories involve unpredictable ATC altitude steps and speed brakes.

### 7. Limitations
Model uncertainty widens during non-standard step-climb phases.

### 8. Next Step
Apply TreeSHAP for global feature attribution and individual flight explanations in 09_shap_explainability.ipynb.
